# Smoke Test — Diagnose Where Eval Hangs

**Purpose:** ~5 min budgeted run that prints timing at every step. Last full eval hung at the inference loop with zero progress for ~67 min. This will tell us exactly *where* the wall is.

**What we measure:**
1. Model load time
2. LoRA wrap time
3. Tokenizer call time
4. **First `generate()` call cost** (4 tokens) — isolates kernel-compile / first-call overhead
5. **Second `generate()` call cost** (16 tokens) — should be much faster if first was compile-bound
6. Real held-out problem (1 only) with token-by-token streaming

If step 4 takes >2 min, the slow-path Mamba is the wall. Next move: properly fix the ptxas-blackwell exec-permission issue and use the fast path.

In [ ]:
# Cell 1: Setup — sitedir discovery + ptxas workaround + (NOTE: NOT disabling fast path this time)
import os, shutil, site, stat, sys, time

T0 = time.perf_counter()
def step(label):
    elapsed = time.perf_counter() - T0
    print(f"[{elapsed:7.2f}s] {label}", flush=True)

step("Starting setup")

# sitedir discovery — same as eval harness v2
for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

# ptxas-blackwell workaround — copy to /tmp with +x
PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        step(f"ptxas-blackwell copied to {PTXAS_DST}")
        break
else:
    step("WARNING: ptxas-blackwell not found at expected paths")

# THIS TIME: do NOT disable fast path — we want to know if ptxas copy alone makes fast path work
step("Setup complete (NOT disabling fast path — testing ptxas binary fix alone)")

In [ ]:
# Cell 2: Load model + tokenizer + apply LoRA (timed)
import kagglehub
import mamba_ssm  # noqa: F401
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

step("Imports done")

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
step(f"kagglehub.model_download returned: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
step("Tokenizer loaded")

t_load = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
step(f"Model from_pretrained done in {time.perf_counter() - t_load:.1f}s")

model.eval()
step("model.eval() done")

lora_cfg = LoraConfig(
    r=32, lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
step("LoRA wrapped (untrained — adapter contributes 0)")

# Probe: is fast path on?
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        step(f"is_fast_path_available = {mod.is_fast_path_available} (in {name})")
        break

step(f"GPU: {torch.cuda.get_device_name(0)}")
step(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 3: Warmup #1 — TINY generate (4 tokens) on a SHORT prompt
# Isolates first-call kernel-compile / autotune cost from anything prompt-length-related.
step("=== Warmup #1: 4 tokens, short prompt ===")

prompt1 = "The capital of France is"
inputs1 = tokenizer(prompt1, return_tensors="pt").to(model.device)
step(f"Tokenized: {inputs1.input_ids.shape}")

t_gen = time.perf_counter()
with torch.no_grad():
    out1 = model.generate(
        **inputs1, max_new_tokens=4, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
step(f"generate() returned in {time.perf_counter() - t_gen:.1f}s")
step(f"Output: {tokenizer.decode(out1[0], skip_special_tokens=True)!r}")

In [ ]:
# Cell 4: Warmup #2 — should be MUCH faster if Cell 3 was compile-bound
step("=== Warmup #2: 16 tokens, short prompt ===")

prompt2 = "List three primary colors:"
inputs2 = tokenizer(prompt2, return_tensors="pt").to(model.device)

t_gen = time.perf_counter()
with torch.no_grad():
    out2 = model.generate(
        **inputs2, max_new_tokens=16, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
step(f"generate() returned in {time.perf_counter() - t_gen:.1f}s")
step(f"Output: {tokenizer.decode(out2[0], skip_special_tokens=True)!r}")

# Per-token estimate
n_new_tokens = out2.shape[1] - inputs2.input_ids.shape[1]
if n_new_tokens > 0:
    step(f"~{(time.perf_counter() - t_gen) / n_new_tokens * 1000:.0f} ms/token after warmup")

In [ ]:
# Cell 5: ONE real held-out problem with TextStreamer (token-by-token printout)
import pandas as pd
from transformers import TextStreamer

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV)
df_sorted = df.sort_values("id").reset_index(drop=True)
row = df_sorted.iloc[-1]  # last row by id sort = first held-out problem

step(f"=== Real problem: id={row['id']} ===")
print(f"PROMPT (first 200 chars): {row['prompt'][:200]!r}")
print(f"GOLD: {row['answer']!r}")

inputs3 = tokenizer(row["prompt"], return_tensors="pt").to(model.device)
step(f"Tokenized: {inputs3.input_ids.shape}")

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

t_gen = time.perf_counter()
with torch.no_grad():
    out3 = model.generate(
        **inputs3, max_new_tokens=128, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, streamer=streamer,
    )
elapsed = time.perf_counter() - t_gen
n_new = out3.shape[1] - inputs3.input_ids.shape[1]
step(f"generate() done in {elapsed:.1f}s ({n_new} tokens, ~{elapsed/max(n_new,1)*1000:.0f} ms/token)")

In [ ]:
# Cell 6: Summary
step("=== Smoke test complete ===")
step(f"Total elapsed: {(time.perf_counter() - T0) / 60:.1f} min")
step("If Cell 3 took >>>Cell 4, first-call kernel compile is the slow part (acceptable, just amortize)")
step("If both warmups were fast and Cell 5 was slow, prompt length is the cost driver")
step("If everything was glacial, slow-path Mamba is the wall — must fix fast path properly")